In [1]:
import os
import tensorflow as tf

# 1. Force TensorFlow to only see the GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# 2. Enable Memory Growth (Prevents "Out of Memory" errors on RTX 3050)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU is active and memory growth is enabled.")
    except RuntimeError as e:
        print(f"GPU Error: {e}")
else:
    print("WARNING: GPU not found. Training will be VERY slow on CPU.")

GPU is active and memory growth is enabled.


In [2]:
import cv2
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [3]:
with tf.device('/GPU:0'):
    input_dir = "LeafData"
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.endswith((".png", ".jpg", ".jpeg")):
                path = os.path.join(root, file)
                img = Image.open(path).convert("RGB")
                img.save(path)  # overwrite with RGB version

c:\Users\Mounika Kola\AppData\Local\Programs\Python\Python310\lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


In [4]:
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_data = datagen.flow_from_directory(
    "LeafData",
    target_size=(224,224),
    color_mode="rgb",
    batch_size=32,
    class_mode="binary",
    subset="training"
)


val_data = datagen.flow_from_directory(
    "LeafData",
    target_size=(224, 224),
    batch_size=32,
    class_mode="binary",
    subset="validation"
)


Found 30556 images belonging to 2 classes.
Found 7639 images belonging to 2 classes.


In [5]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    MaxPooling2D(pool_size=(2,2)),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(pool_size=(2,2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # Binary output: Leaf / Not Leaf
])

In [6]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [7]:
with tf.device('/GPU:0'):
    history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
    )
model.save("leaf_detector.h5")

Epoch 1/10
955/955 [==============================] - 767s 786ms/step - loss: 0.0980 - accuracy: 0.9724 - val_loss: 0.0501 - val_accuracy: 0.9827
Epoch 2/10
955/955 [==============================] - 142s 148ms/step - loss: 0.0504 - accuracy: 0.9861 - val_loss: 0.1441 - val_accuracy: 0.9289
Epoch 3/10
955/955 [==============================] - 139s 145ms/step - loss: 0.0400 - accuracy: 0.9885 - val_loss: 0.0454 - val_accuracy: 0.9855
Epoch 4/10
955/955 [==============================] - 136s 142ms/step - loss: 0.0358 - accuracy: 0.9906 - val_loss: 0.0207 - val_accuracy: 0.9914
Epoch 5/10
955/955 [==============================] - 136s 142ms/step - loss: 0.0278 - accuracy: 0.9920 - val_loss: 0.0405 - val_accuracy: 0.9852
Epoch 6/10
955/955 [==============================] - 137s 143ms/step - loss: 0.0234 - accuracy: 0.9934 - val_loss: 0.0358 - val_accuracy: 0.9915
Epoch 7/10
955/955 [==============================] - 136s 143ms/step - loss: 0.0175 - accuracy: 0.9950 - val_loss: 0.0400 -

In [8]:
from keras.models import load_model

# Load the model you saved 
model = load_model("leaf_detector.h5")

In [9]:
def preprocess_image(img_path):
    img = Image.open(img_path).convert("RGB")   # forces RGB even if grayscale
    img = img.resize((224,224))
    img_array = np.array(img) / 255.0
    return np.expand_dims(img_array, axis=0)

In [10]:
from tensorflow.keras.preprocessing import image
import numpy as np

def is_leaf(img_path):
    img = image.load_img(img_path, target_size=(224,224))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)
    if prediction[0][0] > 0.5:
        return "Leaf"
    else:
        return "Not Leaf"